In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# (선택) 한글 폰트
try:
    import koreanize_matplotlib  # noqa
except:
    pass

# 파일 경로(같은 폴더에 있으면 파일명만)
HEAT_PATH = "merged_heatwave_day_daegu.csv"
MED_PATH  = "대구_우울증&불면증_환자수_전월1.csv"

# -----------------------------
# 1) 기상 데이터: 일별 -> 연도별 열대야일수(5~9월 합계)
# -----------------------------
heat = pd.read_csv(HEAT_PATH, encoding="utf-8")
heat["일시"] = pd.to_datetime(heat["일시"])
heat["year"] = heat["일시"].dt.year
heat["month"] = heat["일시"].dt.month

# 열대야(O/X) => O면 1, 아니면 0
heat["tropical"] = (heat["열대야(O/X)"].astype(str).str.strip() == "O").astype(int)

# 5~9월만
heat = heat[heat["month"].between(5, 9)].copy()

# 연도별 열대야일수 합계
tropical_year = (heat.groupby("year", as_index=False)["tropical"]
                    .sum()
                    .rename(columns={"tropical": "열대야일수"}))

# -----------------------------
# 2) 의료 데이터: 연월 -> 연도별 불면증/우울증 환자수(5~9월 합계)
# -----------------------------
med = pd.read_csv(MED_PATH, encoding="utf-8")

# "2020년 05월" -> year, month
ym = (med["진료년월"].astype(str)
      .str.replace("년", "")
      .str.replace("월", "")
      .str.strip())
med["year"] = ym.str.split().str[0].astype(int)
med["month"] = ym.str.split().str[1].astype(int)

for c in ["우울증_환자수", "불면증_환자수"]:
    med[c] = pd.to_numeric(med[c], errors="coerce")

# 5~9월만
med = med[med["month"].between(5, 9)].copy()

# 연도별(5~9월) 합계
med_year = (med.groupby("year", as_index=False)[["불면증_환자수", "우울증_환자수"]]
              .sum())

# -----------------------------
# 3) 연도 단위로 병합(2020~2025)
# -----------------------------
df_year = pd.merge(tropical_year, med_year, on="year", how="inner")
df_year = df_year[(df_year["year"] >= 2020) & (df_year["year"] <= 2025)].sort_values("year")

# -----------------------------
# 4) 그래프 A: 열대야 연도별 추이
# -----------------------------
plt.figure()
plt.plot(df_year["year"], df_year["열대야일수"], marker="o")
plt.title("대구 연도별 열대야일수(5~9월 합계, 2020~2025)")
plt.xlabel("연도")
plt.ylabel("열대야일수(일)")
plt.xticks(df_year["year"])
plt.grid(True, alpha=0.3)
plt.show()

# -----------------------------
# 5) 그래프 B: 불면증/우울증 연도별 추이
# -----------------------------
plt.figure()
plt.plot(df_year["year"], df_year["불면증_환자수"], marker="o", label="불면증 환자수")
plt.plot(df_year["year"], df_year["우울증_환자수"], marker="o", label="우울증 환자수")
plt.title("대구 연도별 정신질환 환자수(5~9월 합계, 2020~2025)")
plt.xlabel("연도")
plt.ylabel("환자수(명)")
plt.xticks(df_year["year"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# -----------------------------
# 6) 그래프 C: 2020=100 지수화(증가 추세를 한 화면에서 비교)
# -----------------------------
df_idx = df_year.set_index("year")[["열대야일수", "불면증_환자수", "우울증_환자수"]].copy()
df_idx = df_idx / df_idx.loc[2020] * 100  # 2020년을 100으로 변환
df_idx = df_idx.reset_index()

plt.figure()
plt.plot(df_idx["year"], df_idx["열대야일수"], marker="o", label="열대야일수(2020=100)")
plt.plot(df_idx["year"], df_idx["불면증_환자수"], marker="o", label="불면증 환자수(2020=100)")
plt.plot(df_idx["year"], df_idx["우울증_환자수"], marker="o", label="우울증 환자수(2020=100)")
plt.title("기후변화(열대야)와 정신질환 증가 추세 비교 (2020=100)")
plt.xlabel("연도")
plt.ylabel("지수(2020=100)")
plt.xticks(df_idx["year"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(df_year)
